## Практическая работа

В этой практической работе четыре обязательные задачи.

*Обязательные задачи* нужно сделать для того, чтобы проверить, что вы действительно усвоили материал модуля. Сдайте их на проверку.

Удачи!

Цели практической работы:


1.   Потренироваться в обучении моделей деревьев решений.
2.   Потренироваться в обучении моделей случайного леса.
3.   Научиться оценивать качество моделей с помошью Accuracy и confusion matrix.
4.   Научиться увеличивать качество моделей с помощью тюнинга параметров.




Что оценивается:

*   Все пункты и критерия приёмки задания выполнены.
*   Удаление колонок по результатам feature_importance и определения типов производится с помощью кода, а не перечислением их вручную.




## Обязательные задачи

### Описание датасета:
- `id`: идентификатор записи;
- `is_manufacturer_name`: признак производителя автомобиля;

- `region_*`: регион;
- `x0_*`: тип топлива;
- `manufacturer_*`: производитель;
- `short_model_*`: сокращённая модель автомобиля;
- `title_status_*`: статус;
- `transmission_*`: коробка передач;
- `state_*`: штат;
- `age_category_*`: возрастная категория автомобиля;

- `std_scaled_odometer`: количество пройденных миль (после стандартизации);
- `year_std`: год выпуска (после стандартизации);
- `lat_std`: широта (после стандартизации);
- `long_std`: долгота (после стандартизации);
- `odometer/price_std`: отношение стоимости к пробегу автомобиля (после стандартизации);
- `desc_len_std`: количество символов в тексте объявления о продаже (после стандартизации);
- `model_in_desc_std`: количество наименований модели автомобиля в тексте объявления о продаже (после стандартизации);
- `model_len_std`: длина наименования автомобиля (после стандартизации);
- `model_word_count_std`: количество слов в наименовании автомобиля (после стандартизации);
- `month_std`: номер месяца размещения объявления о продаже автомобиля (после стандартизации);
- `dayofweek_std`: день недели размещения объявления о продаже автомобиля (после стандартизации);
- `diff_years_std`: количество лет между годом производства автомобиля и годом размещения объявления о продаже автомобиля (после стандартизации);

- `price`: стоимость;
- `price_category`: категория цены.

0. *Подготовка базовой модели*

Обучите простую модель классификации с помощью DecisionTreeClassifier на данных из датасета vehicles_dataset_prepared.csv. Для этого сделайте следующие шаги:

1. Обучите модель дерева решений с зафиксированным random_state на тренировочной выборке.
2. Сделайте предикт на тестовой выборке.
3. Замерьте точность на тестовой выборке и выведите матрицу ошибок.
4. Удалите фичи с нулевыми весами по feature_importance из тренировочной и тестовой выборок.
5. Заново обучите модель и измерьте качество.

In [3]:
from google.colab import files
uploaded = files.upload()

Saving vehicles_dataset_prepared.csv to vehicles_dataset_prepared.csv


In [4]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [5]:
df = pd.read_csv('vehicles_dataset_prepared.csv')

df_prepared = df.copy()
df_prepared = df_prepared.drop(['price', 'odometer/price_std'], axis=1)

x = df_prepared.drop(['price_category'], axis=1)
y = df_prepared['price_category']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [6]:
# обучение базовой модели
dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train, y_train)

# предсказание
y_pred = dt.predict(x_test)

# метрики
print("Accuracy до удаления фичей:", accuracy_score(y_test, y_pred))
print("Confusion matrix до удаления фичей:\n", confusion_matrix(y_test, y_pred))

# удаление фичей с нулевой важностью
importances = dt.feature_importances_
zero_importance_features = x_train.columns[importances == 0]

print("Количество фичей с нулевой важностью:", len(zero_importance_features))

x_train_reduced = x_train.drop(columns=zero_importance_features)
x_test_reduced = x_test.drop(columns=zero_importance_features)

print("Размеры после удаления фичей:")
print("Train X:", x_train_reduced.shape)
print("Test X:", x_test_reduced.shape)

# повторное обучение модели
dt_reduced = DecisionTreeClassifier(random_state=42)
dt_reduced.fit(x_train_reduced, y_train)

y_pred_reduced = dt_reduced.predict(x_test_reduced)

print("Accuracy после удаления фичей:", accuracy_score(y_test, y_pred_reduced))
print("Confusion matrix после удаления фичей:\n", confusion_matrix(y_test, y_pred_reduced))

Accuracy до удаления фичей: 0.6704781704781705
Confusion matrix до удаления фичей:
 [[738  54 205]
 [ 46 688 219]
 [198 229 509]]
Количество фичей с нулевой важностью: 1096
Размеры после удаления фичей:
Train X: (6733, 364)
Test X: (2886, 364)
Accuracy после удаления фичей: 0.6725571725571725
Confusion matrix после удаления фичей:
 [[730  54 213]
 [ 42 696 215]
 [192 229 515]]


1. *Подготовка модели случайного леса*

Обучите простую модель классификации с помощью RandomForestClassifier. Для этого сделайте следующие шаги:
1. На новых урезанных семплах тренировочной и тестовой выборок обучите модель случайного леса с зафиксированным random_state=50.

2. Сделайте предикт и посчитайте точность модели и матрицу ошибок. Сравните с предыдущей моделью дерева решений. Есть ли случаи, когда модель из пункта 1 отрабатывает лучше, чем модель случайного леса?

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# обучение модели случайного леса
rf = RandomForestClassifier(random_state=50)
rf.fit(x_train_reduced, y_train)

# предсказание
y_pred_rf = rf.predict(x_test_reduced)

# метрики
acc_rf = accuracy_score(y_test, y_pred_rf)
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Accuracy Random Forest:", acc_rf)
print("Confusion Matrix Random Forest:\n", cm_rf)

Accuracy Random Forest: 0.7418572418572419
Confusion Matrix Random Forest:
 [[793  37 167]
 [ 16 789 148]
 [165 212 559]]


In [ ]:
# accuracy: 0.741857
# на 7 процентных пунктов выше, чем у Decision Tree (0.6726)

In [ ]:
# конкретно в этом случае Random Forest почти гарантированно даст более высокую точность и более ровную матрицу ошибок

In [ ]:
# случаи, когда Decision Tree лучше возможны, если:
# 1) данных очень мало
# 2) классы сильно несбалансированы
# 3) случайный лес недонастроен

2. *Тюнинг модели случайного леса*

Увеличьте точность модели на тестовом датасете RandomForestClassifier c помощью тюнинга параметров.

Параметры, которые можно настраивать для увеличения точности:

```
    `bootstrap'
    'max_depth'
    'max_features'
    'min_samples_leaf'
    'min_samples_split'
    'random_state'
    'n_estimators'

```



С описанием каждого из параметров можно ознакомиться в документации:


https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

Задание засчитывается, если значение метрики строго выше 0,76 на тестовом датасете.

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

rf_tuned = RandomForestClassifier(
    n_estimators=5000,
    max_depth=None,
    max_features='log2',
    min_samples_split=2,
    min_samples_leaf=1,
    bootstrap=True,
    class_weight='balanced_subsample',
    random_state=50,
    n_jobs=-1,
    warm_start=False
)

rf_tuned.fit(x_train_reduced, y_train)
y_pred_tuned = rf_tuned.predict(x_test_reduced)

print("Accuracy после тюнинга:", accuracy_score(y_test, y_pred_tuned))
print("Confusion Matrix после тюнинга:\n", confusion_matrix(y_test, y_pred_tuned))

Accuracy после тюнинга: 0.7605682605682605
Confusion Matrix после тюнинга:
 [[808  38 151]
 [ 14 808 131]
 [153 204 579]]


3. *Анализ влияния фичей на модель*

Во всех задачах до вы работали над подготовленным датасетом, где фичи были заранее извлечены из текстовых переменных, отскейлены и пропущены через One Hot Encoder. Сравним, какой была бы предсказательная способность модели, если бы мы использовали только сырые данные из исходного датасета. Для этого проделайте следующие шаги:

1. Загрузите датасет `vehicles_dataset_old.csv`.
2. Удалите из него переменную `price` и все строковые колонки. Дерево решений и случайный лес не умеют самостоятельно работать со строковыми значениями.
3. Сформируйте x_train и x_test так же, как они были сформированы в предыдущих заданиях.
4. Обучите свою лучшую модель случайного леса на новых данных и замерьте качество. Убедитесь, что оно ухудшилось.
5. Найдите три фичи, которые лучшим образом повлияли на предсказательную способность модели.

In [17]:
from google.colab import files
uploaded = files.upload()

Saving vehicles_dataset_old.csv to vehicles_dataset_old.csv


In [18]:
df_old = pd.read_csv('vehicles_dataset_old.csv')
df_old.head()

,id,url,region,region_url,price,year,manufacturer,model,fuel,odometer,title_status,transmission,image_url,description,state,lat,long,posting_date,price_category,date
0,7308295377,https://chattanooga.craigslist.org/ctd/d/chatt...,chattanooga,https://chattanooga.craigslist.org,54990,2020,ram,2500 crew cab big horn,diesel,27442,clean,other,https://images.craigslist.org/00N0N_1xMPvfxRAI...,Carvana is the safer way to buy a car During t...,tn,35.060000,-85.250000,2021-04-17T12:30:50-0400,high,2021-04-17 16:30:50+00:00
1,7316380095,https://newjersey.craigslist.org/ctd/d/carlsta...,north jersey,https://newjersey.craigslist.org,16942,2016,ford,explorer 4wd 4dr xlt,other,60023,clean,automatic,https://images.craigslist.org/00x0x_26jl9F0cnL...,***Call Us for more information at: 201-635-14...,nj,40.821805,-74.061962,2021-05-03T15:40:21-0400,medium,2021-05-03 19:40:21+00:00
2,7313733749,https://reno.craigslist.org/ctd/d/atlanta-2017...,reno / tahoe,https://reno.craigslist.org,35590,2017,volkswagen,golf r hatchback,gas,14048,clean,other,https://images.craigslist.org/00y0y_eeZjWeiSfb...,Carvana is the safer way to buy a car During t...,ca,33.779214,-84.411811,2021-04-28T03:52:20-0700,high,2021-04-28 10:52:20+00:00
3,7308210929,https://fayetteville.craigslist.org/ctd/d/rale...,fayetteville,https://fayetteville.craigslist.org,14500,2013,toyota,rav4,gas,117291,clean,automatic,https://images.craigslist.org/00606_iGe5iXidib...,2013 Toyota RAV4 XLE 4dr SUV Offered by: R...,nc,35.715954,-78.655304,2021-04-17T10:08:57-0400,medium,2021-04-17 14:08:57+00:00
4,7303797340,https://knoxville.craigslist.org/ctd/d/knoxvil...,knoxville,https://knoxville.craigslist.org,14590,2012,bmw,1 series 128i coupe 2d,other,80465,clean,other,https://images.craigslist.org/00F0F_5UAXmOzC18...,Carvana is the safer way to buy a car During t...,tn,35.970000,-83.940000,2021-04-08T15:10:56-0400,medium,2021-04-08 19:10:56+00:00


In [23]:
df_old = df_old.drop('price', axis=1)

string_columns = df_old.select_dtypes(include='object').columns

df_old = df_old.drop(columns=string_columns)
display(df_old.head())

,id,year,odometer,lat,long
0,7308295377,2020,27442,35.060000,-85.250000
1,7316380095,2016,60023,40.821805,-74.061962
2,7313733749,2017,14048,33.779214,-84.411811
3,7308210929,2013,117291,35.715954,-78.655304
4,7303797340,2012,80465,35.970000,-83.940000


In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

# перезагрузка исходного датасета, чтобы восстановить price_category
df_old_original = pd.read_csv('vehicles_dataset_old.csv')

# отделение целевой переменной
y_old = df_old_original['price_category']

# удаление price и price_category из признаков
x_old = df_old_original.drop(['price', 'price_category'], axis=1)

# нахождение и удаление всех строковых признаков (только из x)
string_cols_to_drop_from_x = x_old.select_dtypes(include='object').columns
x_old = x_old.drop(columns=string_cols_to_drop_from_x)

# деление данных на обучающую и тестовую выборки, как в предыдущих заданиях
x_train_old, x_test_old, y_train_old, y_test_old = train_test_split(
    x_old, y_old, test_size=0.3, random_state=42
)

In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

rf_tuned_raw = RandomForestClassifier(
    n_estimators=5000,
    max_depth=None,
    max_features='log2',
    min_samples_split=2,
    min_samples_leaf=1,
    bootstrap=True,
    class_weight='balanced_subsample',
    random_state=50,
    n_jobs=-1,
    warm_start=False
)

# обучение модели на сырых данных
rf_tuned_raw.fit(x_train_old, y_train_old)

# предсказания на тестовой выборке сырых данных
y_pred_raw = rf_tuned_raw.predict(x_test_old)

# оценка качества модели
accuracy_raw = accuracy_score(y_test_old, y_pred_raw)
cm_raw = confusion_matrix(y_test_old, y_pred_raw)

print("Accuracy на сырых данных:", accuracy_raw)
print("Confusion Matrix на сырых данных:\n", cm_raw)

Accuracy на сырых данных: 0.6403326403326404
Confusion Matrix на сырых данных:
 [[705  76 216]
 [ 48 711 194]
 [268 236 432]]


In [ ]:
# подготовленные данные:
# accuracy = 0.76

# сырые данные:
# accuracy = 0.64

In [35]:
feature_importances_raw = rf_tuned_raw.feature_importances_
feature_names_raw = x_train_old.columns

importances_series_raw = pd.Series(feature_importances_raw, index=feature_names_raw)

top_3_features_raw = importances_series_raw.nlargest(3)

print("Топ-3 фичи, повлиявшие на предсказательную способность модели на сырых данных:")
for feature, importance in top_3_features_raw.items():
    print(f"{feature}: {importance:.6f}")

Топ-3 фичи, повлиявшие на предсказательную способность модели на сырых данных:
odometer: 0.271747
year: 0.231798
id: 0.167074
